# 01 - Fairness fundamentals

**What you'll learn**
- What *privileged / unprivileged* groups are and how `callmefair` encodes them
- How to split data and train a model with `BMInterface`
- How the fairness metrics SPD and DI are computed (by hand, then with `BMMetrics`)
- How five metrics are combined into one fairness score
- Why the choice of privileged group changes the sign of a metric

**Prerequisite:** [00_data_and_preprocessing.ipynb](00_data_and_preprocessing.ipynb) (how the data was cleaned and how groups are defined). **Runtime:** about 1 minute.

## 1. Setup

In [1]:
from fhg_examples import setup
setup()

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from callmefair import BMInterface, BMMetrics, calculate_fairness_score
from fhg_examples.datasets import load, SPECS

## 2. The data and the two groups

We use the *Diabetes 130-US hospitals* data. The label `readmitted` is 1 when the patient was readmitted within 30 days.
Fairness is always measured with respect to a **sensitive attribute**, here `race`. `callmefair` expects it to be binary:
value **1 = privileged group** (Caucasian), value **0 = unprivileged group** (all other races).
"Privileged" is a convention for the reference group, not a judgement. See `00_data_and_preprocessing.ipynb` for how these groups were built.

In [2]:
df = load('diabetes')
spec = SPECS['diabetes']
privileged_group, unprivileged_group = spec.attribute('race').groups
print(privileged_group, unprivileged_group)
print(df.shape)

[{'race': 1}] [{'race': 0}]
(99492, 67)


A first look at the groups: their size and their **base rate** (share of patients with label 1). If base rates differ a lot, even a perfect model would produce different positive rates.

In [3]:
groups = df.groupby('race')['readmitted'].agg(patients='size', base_rate='mean')
groups.index = groups.index.map({1: '1 (privileged)', 0: '0 (unprivileged)'})
groups

,patients,base_rate
race,,
0 (unprivileged),23393,0.110
1 (privileged),76099,0.113


## 3. Split the data with `BMInterface`

`BMInterface` makes a stratified **train / validation / test** split (80 / 10 / 10 by default) and wraps each part as an AIF360 dataset.
The model learns on *train*, the decision threshold is chosen on *validation*, and all fairness numbers are measured on *test*.

In [4]:
bmI = BMInterface(df, label='readmitted', protected=['race'], random_state=42)
X_train, y_train = bmI.get_train_xy()
X_val, _ = bmI.get_val_xy()
X_test, y_test = bmI.get_test_xy()
pd.Series({'train': len(X_train), 'validation': len(X_val), 'test': len(X_test)}, name='rows')

train         79592
validation     9950
test           9950
Name: rows, dtype: int64

## 4. Train a model and predict probabilities

Any classifier that returns probabilities works. A logistic regression is fast and enough for learning the concepts.

In [5]:
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
pred_val = model.predict_proba(X_val)
pred_test = model.predict_proba(X_test)
pred_test[:3].round(3)   # columns follow model.classes_ = [0, 1]

array([[0.947, 0.053],
       [0.79 , 0.21 ],
       [0.8  , 0.2  ]])

## 5. Two fairness metrics by hand

Turn probabilities into decisions with a threshold, then compare the **positive rate** (share of patients predicted 1) of the two groups.
For a fixed threshold of 0.5:

- **SPD** (statistical parity difference) = rate(unprivileged) - rate(privileged). Ideal **0**; acceptable within (-0.1, 0.1).
- **DI** (disparate impact) = rate(unprivileged) / rate(privileged). Ideal **1**; acceptable within (0.8, 1.2).

In [6]:
test_race = bmI.get_test_BLD().protected_attributes.ravel()   # 1 = privileged, 0 = unprivileged
decision = pred_test[:, 1] > 0.5
rate_priv = decision[test_race == 1].mean()
rate_unpriv = decision[test_race == 0].mean()
print(f'positive rate  privileged={rate_priv:.3f}  unprivileged={rate_unpriv:.3f}')
print(f'SPD = {rate_unpriv - rate_priv:.3f}    DI = {rate_unpriv / rate_priv:.3f}')

positive rate  privileged=0.005  unprivileged=0.004
SPD = -0.001    DI = 0.821


With a 0.5 threshold on this imbalanced data (only ~11% positives) almost nobody is predicted positive, so the numbers are not meaningful.
That is why `BMMetrics` does not use 0.5 but the threshold that maximises **balanced accuracy** on the validation set.

## 6. The same with `BMMetrics`

`BMMetrics` receives the validation and test probabilities, the `class_array` (`model.classes_`) and the two groups. It searches the threshold on validation and computes every metric on test.

In [7]:
bmM = BMMetrics(bmI, model.classes_, pred_val, pred_test, privileged_group, unprivileged_group)
print('threshold chosen on validation:', round(float(bmM.best_class_thresh), 3))

threshold chosen on validation: 0.109


The curves show why the threshold matters: balanced accuracy (top) and SPD (bottom) both change with it, and the dotted line marks the value picked on validation. The green band is the acceptable SPD range.

In [ ]:
from callmefair.viz import plot_threshold_sweep
plot_threshold_sweep(bmM, metric='spd')

To verify our understanding, recompute SPD and DI by hand using that same threshold. They match `get_report()` in the next step.

In [8]:
decision = pred_test[:, 1] > bmM.best_class_thresh
rate_priv, rate_unpriv = decision[test_race == 1].mean(), decision[test_race == 0].mean()
print(f'by hand: SPD = {rate_unpriv - rate_priv:.3f}   DI = {rate_unpriv / rate_priv:.3f}')

by hand: SPD = -0.020   DI = 0.948


The bars below show the three group rates behind the metrics. The gap between the blue and orange bar in the first group is exactly the SPD computed above.

In [ ]:
from callmefair.viz import plot_group_rates
plot_group_rates(bmM)

## 7. The report: performance vs fairness

`get_report()` returns a dictionary. We split it into two tables: how *accurate* the model is, and how *fair* it is.

| Fairness metric | Meaning | Ideal | Acceptable |
|---|---|---|---|
| `eq_opp_diff` (EOD) | true-positive-rate gap between groups (unprivileged - privileged) | 0 | (-0.1, 0.1) |
| `avg_odd_diff` (AOD) | average of the true- and false-positive-rate gaps | 0 | (-0.1, 0.1) |
| `spd` (SPD) | gap in positive prediction rate | 0 | (-0.1, 0.1) |
| `disparate_impact` (DI) | ratio of positive prediction rates | 1 | (0.8, 1.2) |
| `theil_idx` (TI) | inequality of "benefit" across individuals | 0 | (0, 0.25) |

In [9]:
report = bmM.get_report()
perf_keys = ['balanced_acc', 'acc', 'precision', 'recall', 'f1', 'mcc']
fair_keys = ['eq_opp_diff', 'avg_odd_diff', 'spd', 'disparate_impact', 'theil_idx']
performance = pd.Series({k: report[k] for k in perf_keys}, name='performance (test set)').to_frame()
fairness = pd.Series({k: report[k] for k in fair_keys}, name='fairness (test set)').to_frame()
display(performance, fairness)

,performance (test set)
balanced_acc,0.603
acc,0.642
precision,0.168
recall,0.553
f1,0.258
mcc,0.035


,fairness (test set)
eq_opp_diff,-0.035
avg_odd_diff,-0.025
spd,-0.020
disparate_impact,0.948
theil_idx,0.110


The `spd` and `disparate_impact` values agree with our hand computation above.

**Fair range view.** Each fairness metric is drawn against its acceptable range (green band): filled points are inside, hollow points outside.

In [ ]:
from callmefair.viz import plot_fair_ranges
baseline = [{**report, 'fair_score': bmM.get_score()['overall_score'], 'model': 'LogisticRegression', 'BM': 'baseline'}]
plot_fair_ranges(baseline)

## 8. One number: the fairness score

Five metrics are hard to compare across models, so `calculate_fairness_score` merges them into a single score **between 0 and 1**
(**0 = perfectly fair, higher = more biased**). Each metric adds a penalty if it is outside its acceptable range, plus a penalty
proportional to its distance from the ideal. `bmM.get_score()` is a shortcut for passing the report's metrics.

In [10]:
score = calculate_fairness_score(EOD=report['eq_opp_diff'], AOD=report['avg_odd_diff'],
                                 SPD=report['spd'], DI=report['disparate_impact'],
                                 TI=report['theil_idx'])
print('overall_score:', score['overall_score'], '| is_fair:', score['is_fair'])
pd.DataFrame({'value': pd.Series({k: report[m] for k, m in zip(['EOD','AOD','SPD','DI','TI'], fair_keys)}),
              'in_range': score['metric_evaluations'], 'deviation': score['deviations']})

overall_score: 0.133 | is_fair: True


,value,in_range,deviation
EOD,-0.035,True,0.353
AOD,-0.025,True,0.249
SPD,-0.020,True,0.195
DI,0.948,True,0.261
TI,0.110,True,0.440


`deviation` is the distance from the ideal value in units of the acceptable half-width (1.0 means exactly on the edge of the acceptable range). `is_fair` is true only when *all* five metrics are in range.

## 9. Which group is privileged matters

SPD and DI are differences/ratios *between* the two groups, so swapping the roles flips SPD's sign (and inverts DI).
The size of the gap is essentially unchanged (the score barely moves), because the score uses absolute deviations from the ideal.

In [11]:
swapped = BMMetrics(bmI, model.classes_, pred_val, pred_test, unprivileged_group, privileged_group)
rows = {'race=1 privileged': bmM.get_report(), 'race=0 privileged': swapped.get_report()}
pd.DataFrame(rows).loc[['spd', 'disparate_impact']].T.assign(
    score=[bmM.get_score()['overall_score'], swapped.get_score()['overall_score']])

,spd,disparate_impact,score
race=1 privileged,-0.02,0.948,0.133
race=0 privileged,0.02,1.055,0.134


## Summary
- A **sensitive attribute** splits patients into a privileged (1) and an unprivileged (0) group; fairness compares the model's behaviour on them.
- `BMInterface` provides train / validation / test sets; the threshold is tuned on validation and metrics are measured on test.
- SPD is a difference of positive rates (ideal 0), DI is their ratio (ideal 1); `BMMetrics.get_report()` returns these plus EOD, AOD and TI.
- `calculate_fairness_score` condenses the five metrics into one number in [0, 1] where **lower is fairer**.
- Swapping the privileged group flips the sign of SPD and inverts DI, but not the size of the bias.

**Next:** [02_bias_search.ipynb](02_bias_search.ipynb): compare several sensitive attributes automatically to decide which one to mitigate.